In [3]:
import json, re
from typing import List, Dict

def load_dataset(path: str) -> List[Dict]:
    with open(path) as f:
        return json.load(f)

def clean_text(text: str) -> str:
    text = re.sub(r"[•·▪◦\-–—]\s*", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
#get job description
def extract_samples(dataset: List[Dict]) -> List[Dict]:
    samples = []
    for job in dataset:
        jd = clean_text(job.get("job_description", "").lower())
        goal = set(s.lower() for s in job.get("it_skills", []))
        samples.append({"text": jd, "goal_skills": goal, "employer": job.get("employer_name")})
    return samples
def clean_bert_word(word: str) -> str:
    word = word.replace(" ##", "")
    word = word.replace("##", "")
    word = word.strip().lower()
    return word

In [ ]:
#build synonym matching
from rapidfuzz import fuzz
def build_synonym_map(dataset, threshold=85): #used to store different form of words (aws, amazon web services)
    all_skills = []

    for job in dataset:
        for skill in job.get("goal_skills", set()):
            s = skill.strip().lower()
            all_skills.append(s)

    unique = list(set(all_skills))

    synonym_map = {}

    for skill in unique:
        synonym_map[skill] = skill

    # fuzzy clustering
    for skill in unique:
        for other in unique:
            if skill == other:
                continue

            if fuzz.ratio(skill, other) >= threshold:
                canonical = min([skill, other], key=len)
                synonym_map[skill] = canonical
                synonym_map[other] = canonical

   
    synonym_map.update({
        "reactjs": "react",
        "react.js": "react",
        "react js": "react",
        "nodejs": "node",
        "node.js": "node",
        "node js": "node",
        "javascript": "js",
        "typescript": "ts",
        "aws lambda": "aws",
        "amazon web services": "aws",
        "postgresql": "postgres",
        "scikit-learn": "sklearn",
        "scikit learn": "sklearn",
        "artificial intelligence": "ai",
        "large language model": "llm"
    })

    return synonym_map

def normalize_skill(skill: str, synonym_map: Dict[str, str]) -> str:
    skill = skill.strip().lower()
    return synonym_map.get(skill, skill)


In [ ]:
import spacy
nlp = spacy.blank("en")

def find_skill_spans(text: str, goal_skills: set, synonym_map: Dict) -> List[Dict]: 
    """
    Returns list of {start, end, text, label} where text[start:end] is a skill mention.
    Handles multi-word skills like 'machine learning', 'spring boot'.
    """
    text_lower = text.lower()
    spans = []
    seen = set()

    # longer phrase match first
    sorted_skills = sorted(goal_skills, key=len, reverse=True)

    for skill in sorted_skills:
        canonical = normalize_skill(skill, synonym_map)
        search_terms = {skill, canonical}  # try both forms
        for term in search_terms:
            idx = 0
            while True:
                pos = text_lower.find(term, idx)
                if pos == -1:
                    break
                end = pos + len(term)
                # Boundary check: must be word-bounded
                before_ok = pos == 0 or not text_lower[pos-1].isalnum()
                after_ok  = end == len(text) or not text_lower[end].isalnum()
                if before_ok and after_ok:
                    key = (pos, end)
                    if key not in seen:
                        seen.add(key)
                        spans.append({"start": pos, "end": end,
                                      "span_text": text[pos:end], "label": canonical})
                idx = pos + 1

    # Remove nested spans (keep longest)
    spans.sort(key=lambda s: s["end"] - s["start"], reverse=True)
    clean = []
    covered = set()
    for s in spans:
        positions = set(range(s["start"], s["end"]))
        if not positions & covered:
            clean.append(s)
            covered |= positions
    return sorted(clean, key=lambda s: s["start"])


def bio_tag_sentence(text: str, spans: List[Dict]) -> List[Dict]:
    """
    Returns [{token: str, start: int, end: int, bio: str}, ...]
    """
    text = text.lower()
    doc = nlp(text)
    tokens = []
    for tok in doc:
        label = "O"
        for span in spans:
            if tok.idx >= span["start"] and tok.idx < span["end"]:
                label = "B-SKILL" if tok.idx == span["start"] else "I-SKILL"
                break
        tokens.append({"token": tok.text, "start": tok.idx,
                        "end": tok.idx + len(tok.text), "bio": label})
    return tokens

# Build full training set
def build_training_data(samples, synonym_map):
    training = []
    for s in samples:
        spans = find_skill_spans(s["text"], s["goal_skills"], synonym_map)
        tagged = bio_tag_sentence(s["text"], spans)
        # Split into sentences for tractable sequence length
        sentences = split_into_sentences(s["text"], tagged)
        training.extend(sentences)
    return training

def split_into_sentences(text, tagged_tokens, max_len=128):
    """Chunk by sentence boundaries, max 128 tokens for DistilBERT."""
    sentences = []
    chunk, chunk_labels = [], []
    for tok in tagged_tokens:
        chunk.append(tok["token"])
        chunk_labels.append(tok["bio"])
        if tok["token"] in {".", "!", "?"} or len(chunk) >= max_len:
            if chunk:
                sentences.append({"tokens": chunk, "labels": chunk_labels})
            chunk, chunk_labels = [], []
    if chunk:
        sentences.append({"tokens": chunk, "labels": chunk_labels})
    return sentences


In [ ]:
from transformers import (DistilBertTokenizerFast,
                           DistilBertForTokenClassification, TrainingArguments)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

import torch, numpy as np

LABEL_LIST = ["O", "B-SKILL", "I-SKILL"]
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True, max_length=128,
        is_split_into_words=True,  # tokens already split
        padding="max_length"
    )
    all_labels = []
    for i, label_seq in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)          # CLS, SEP, PAD → ignore
            elif wid != prev_word:
                label_ids.append(LABEL2ID[label_seq[wid]])
            else:
                # continuation sub-token: B→I, keep I
                raw = label_seq[wid]
                label_ids.append(LABEL2ID["I-SKILL"] if raw == "B-SKILL"
                                 else LABEL2ID[raw])
            prev_word = wid
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

# Build HF Dataset
def make_hf_dataset(training_data):
    raw = {"tokens": [s["tokens"] for s in training_data],
           "labels": [s["labels"] for s in training_data]}
    ds = Dataset.from_dict(raw)
    return ds.map(tokenize_and_align, batched=True,
                  remove_columns=["tokens", "labels"])

# Load model
model = DistilBertForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Seqeval metrics
from seqeval.metrics import classification_report, f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id != -100:
                current_preds.append(ID2LABEL[pred_id])
                current_labels.append(ID2LABEL[label_id])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

args = TrainingArguments(
    output_dir="./skill-ner",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

def keep_skill_context(sample, window=20):
    tokens = sample["tokens"]
    labels = sample["labels"]

    skill_positions = [
        i for i, lab in enumerate(labels)
        if lab in {"B-SKILL", "I-SKILL"}
    ]

    # If no skill in this sample, optionally drop or keep small part
    if not skill_positions:
        return {
            "tokens": tokens[:40],
            "labels": labels[:40]
        }

    keep = set()

    for pos in skill_positions:
        start = max(0, pos - window)
        end = min(len(tokens), pos + window + 1)
        keep.update(range(start, end))

    keep = sorted(keep)

    return {
        "tokens": [tokens[i] for i in keep],
        "labels": [labels[i] for i in keep]
    }

a = load_dataset("it-skills.json")
b = extract_samples(a)

synonym_map = build_synonym_map(b)
training_data = build_training_data(b,synonym_map)
training_data = [keep_skill_context(sample) for sample in training_data]

train_data, eval_data = train_test_split(training_data, test_size=0.2, random_state=42, shuffle=True)
train_ds = make_hf_dataset(train_data)
eval_ds = make_hf_dataset(eval_data)

from collections import Counter

label_counter = Counter()

for sample in training_data:
    label_counter.update(sample["labels"])

print(label_counter)
total = sum(label_counter.values())




Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2842 [00:00<?, ? examples/s]

Map:   0%|          | 0/711 [00:00<?, ? examples/s]

Counter({'O': 76675, 'B-SKILL': 3159, 'I-SKILL': 831})


In [ ]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import Subset
from transformers import Trainer,DataCollatorForTokenClassification


def undersample_dataset(dataset, label_column="labels", majority_class=0, ratio=1.0):
    """
    Undersample the majority class to balance the dataset.
    """
    all_labels = []
    for item in dataset:
        labels = item[label_column]
        # Flatten and filter out ignored indices (-100)
        if isinstance(labels, torch.Tensor):
            valid = labels[labels != -100].tolist()
        else:
            valid = [l for l in labels if l != -100]
        all_labels.append(valid)

    # Find samples that contain at least one non-majority label
    minority_indices = [
        i for i, labels in enumerate(all_labels)
        if any(l != majority_class for l in labels)
    ]
    majority_only_indices = [
        i for i, labels in enumerate(all_labels)
        if all(l == majority_class for l in labels)
    ]

    # Keep only ratio * |minority| majority samples
    n_majority_keep = int(len(minority_indices) * ratio)
    np.random.shuffle(majority_only_indices)
    kept_majority_indices = majority_only_indices[:n_majority_keep]

    final_indices = minority_indices + kept_majority_indices
    np.random.shuffle(final_indices)

    print(f"Original dataset size: {len(dataset)}")
    print(f"Minority-containing samples: {len(minority_indices)}")
    print(f"Majority-only samples kept: {len(kept_majority_indices)} / {len(majority_only_indices)}")
    print(f"Undersampled dataset size: {len(final_indices)}")

    return Subset(dataset, final_indices)


class TokenClassificationTrainer(Trainer):
    """Standard trainer — loss weighting removed in favour of undersampling."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss


id2label = ID2LABEL

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [id2label[pred] for pred, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions),
        "report": classification_report(true_labels, true_predictions),
    }

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

balanced_train_dataset = undersample_dataset(
    train_ds,
    label_column="labels",
    majority_class=0,
    ratio=2.0         
)

trainer = TokenClassificationTrainer(
    model=model,
    args=args,
    train_dataset=balanced_train_dataset,   # ← undersampled
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

eval_metrics = trainer.evaluate()

print("\nEvaluation metrics:")
for key, value in eval_metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

Original dataset size: 2842
Minority-containing samples: 950
Majority-only samples kept: 1892 / 1892
Undersampled dataset size: 2842


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Report
1,No log,0.082662,0.676322,precision recall f1-score support SKILL 0.66 0.70 0.68 769 micro avg 0.66 0.70 0.68 769 macro avg 0.66 0.70 0.68 769 weighted avg 0.66 0.70 0.68 769
2,No log,0.046365,0.837419,precision recall f1-score support SKILL 0.83 0.84 0.84 769 micro avg 0.83 0.84 0.84 769 macro avg 0.83 0.84 0.84 769 weighted avg 0.83 0.84 0.84 769
3,No log,0.042307,0.875000,precision recall f1-score support SKILL 0.85 0.90 0.88 769 micro avg 0.85 0.90 0.88 769 macro avg 0.85 0.90 0.88 769 weighted avg 0.85 0.90 0.88 769
4,No log,0.041591,0.880407,precision recall f1-score support SKILL 0.86 0.90 0.88 769 micro avg 0.86 0.90 0.88 769 macro avg 0.86 0.90 0.88 769 weighted avg 0.86 0.90 0.88 769
5,No log,0.044564,0.867995,precision recall f1-score support SKILL 0.83 0.91 0.87 769 micro avg 0.83 0.91 0.87 769 macro avg 0.83 0.91 0.87 769 weighted avg 0.83 0.91 0.87 769


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,F1,Report
No log,0.041591,5,0.880407,precision recall f1-score support SKILL 0.86 0.90 0.88 769 micro avg 0.86 0.90 0.88 769 macro avg 0.86 0.90 0.88 769 weighted avg 0.86 0.90 0.88 769



Evaluation metrics:
eval_loss: 0.0416
eval_f1: 0.8804
eval_report:               precision    recall  f1-score   support

       SKILL       0.86      0.90      0.88       769

   micro avg       0.86      0.90      0.88       769
   macro avg       0.86      0.90      0.88       769
weighted avg       0.86      0.90      0.88       769



In [24]:
from transformers import pipeline
from typing import List

ner_pipe = pipeline(
    "token-classification",
    model="./skill-ner/checkpoint-445",  # change to latest/best checkpoint
    tokenizer="./skill-ner/checkpoint-445",
    aggregation_strategy="max"
)

def extract_skills(text: str, threshold: float = 0.3 ) -> List[str]:
    results = ner_pipe(text)
    skills = []

    for ent in results:
        if ent.get("entity_group") == "SKILL" and ent["score"] >= threshold:
            skills.append(ent["word"].strip().lower())
  
    return list(set(skills))
test_text = '''I need candidates that have deep understand in python java sql, machine learning, cloud platforms, gcp '''
print(extract_skills(test_text,0.3))





Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

['sql', 'java', 'machine learning', 'gcp', 'python']
